# LLMOps: The LangChain Framework

In [ ]:
# Day 46 — The LangChain Framework
#
# The OpenAI API (from Day 45) is great for simple text generation.
# But what if you want to:
#   - Read a 100-page PDF and ask questions about it?
#   - Connect the LLM to a SQL database?
#   - Give the LLM "Memory" so it remembers the conversation?
#   - Switch from OpenAI to an open-source model seamlessly?
#
# You need an orchestration framework. That is LangChain.
# LangChain provides the "glue" to connect LLMs to your data and tools.
#
# Today we'll cover:
#   1. PromptTemplates and LLMChains
#   2. Adding Memory to a Chatbot
#   3. Introduction to Agents (LLMs using tools)
#
# Install: pip install langchain langchain-openai

### 1. PromptTemplates and Chains (LCEL)

In [ ]:
# Instead of manually formatting python strings (f"Hello {name}"),
# LangChain uses PromptTemplates to build reusable AI instructions.
# LCEL (LangChain Expression Language) uses the pipe `|` operator to chain things together.

langchain_chain_code = '''from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. The Model
llm = ChatOpenAI(model="gpt-3.5-turbo")

# 2. The Template
template = """
You are a DevOps expert.
Explain the concept of {technology} in {language}.
Keep it under 50 words.
"""
prompt = PromptTemplate.from_template(template)

# 3. The Chain (Prompt -> LLM -> Output as String)
chain = prompt | llm | StrOutputParser()

# 4. Execute the chain with variables
result = chain.invoke({"technology": "Kubernetes", "language": "Spanish"})
print(result)
'''

print("LangChain LCEL Pattern:")
print(langchain_chain_code)

### 2. Giving the LLM Memory

In [ ]:
# LLMs are stateless (they have goldfish memory). 
# If you ask "What is Docker?" and then ask "How do I install it?", 
# the API forgets you were talking about Docker.
# LangChain's Memory components automatically pass the chat history back into the prompt.

langchain_memory_code = '''from langchain_openai import ChatOpenAI
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

llm = ChatOpenAI()

# ConversationBufferMemory stores the entire chat transcript
memory = ConversationBufferMemory()

# The ConversationChain handles appending history to the prompt for you
conversation = ConversationChain(
    llm=llm, 
    memory=memory,
    verbose=True # Set to True to see what LangChain is doing under the hood!
)

print(conversation.predict(input="Hi, I'm an MLOps engineer."))
print(conversation.predict(input="What is my job title?")) # It remembers!
'''

print("LangChain Memory Pattern:")
print(langchain_memory_code)

### 3. Agents (LLMs with Tools)

In [ ]:
# THIS is the most powerful part of LangChain.
# An "Agent" is an LLM that has been given a list of Tools (Python functions).
# The LLM decides WHICH tool to use, writes the inputs, runs it, and reads the output!

agent_code = '''from langchain.agents import initialize_agent, AgentType, Tool
from langchain_openai import ChatOpenAI
import subprocess

# 1. Define a custom tool (a python function)
def check_disk_space(dummy_arg):
    """Runs the 'df -h' command and returns disk usage."""
    result = subprocess.run(['df', '-h'], capture_output=True, text=True)
    return result.stdout

# Wrap the function in a LangChain Tool object
disk_tool = Tool(
    name="DiskSpaceChecker",
    func=check_disk_space,
    description="Useful for finding out how much disk space is left on the server."
)

# 2. Initialize Agent
llm = ChatOpenAI(temperature=0)
tools = [disk_tool]

agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, 
    verbose=True
)

# 3. Run the Agent
# The LLM will read the prompt, realize it needs disk space info, call the tool,
# read the output, and formulate a final answer!
agent.run("Is the server running out of disk space? Analyze the current usage.")
'''

print("LangChain Agent Pattern:")
print(agent_code)

In [ ]:
# Practice Questions:
#
# 1. What does the pipe operator `|` do in LangChain Expression Language (LCEL)?
# 2. Why is Memory important for building Chatbots, and how does ConversationBufferMemory work?
# 3. In an Agent, why is the `description` field of a Tool extremely important for the LLM?

### DevOps / AI-MLOps Practice

In [ ]:
# Exercise 1: The Azure Cloud Agent
# Imagine you create two Python functions:
# - `get_running_vms()` (Uses Azure SDK to list VMs)
# - `restart_vm(vm_name)` (Uses Azure SDK to restart a VM)
# Wrap these in LangChain Tools, and give them to an Agent.
# Now you can type: "Hey, are there any dev VMs running? If so, restart web-01."
# The LLM will autonomously execute your infrastructure code!

# Why this matters for me as a DevOps/MLOps engineer:
# 1. ChatOps: You can build Slack bots (Agents) that SREs can talk to in plain English to fix server issues.
# 2. Abstraction: LangChain lets you swap `ChatOpenAI()` for `ChatAnthropic()` or a local HuggingFace model by changing exactly one line of code.
# 3. Framework Power: Building an Agent from scratch using pure API calls requires hundreds of lines of while-loops and regex. LangChain handles the agent logic automatically.